# 07 RQ1 supporting analysis: event-level forward test and detection delay
**DATA3001 · Group Sushi** · California Current, 2012–2021

Groups OISST heatwave cells into regional events (469 in 2012–2021), chooses the alert rule on events before December 2018, tests it once on the 112 unseen 2019–20 events, and measures the delay between the start of each event and the first drifter alert (Sections 7.4 and 8.1 of the README). Needs `data/gdp_cc_subset.pkl` and `data/oisst/`, which `06_RQ3_species_exposure.py` downloads.


In [1]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy import ndimage, stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import xarray as xr
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ======================= CONFIG =======================
DATA_DIR   = "data"
GDP_FILE   = os.path.join(DATA_DIR, "gdp_cc_subset.pkl")     # written by 03_final_analysis (Part 1)
OISST_DIR  = os.path.join(DATA_DIR, "oisst")                 # oisst_YYYY.nc, one file per year
CUFES_FILE = os.path.join(DATA_DIR, "cufes_cc.csv")          # CalCOFI CUFES egg samples (ERDDAP erdCalCOFIcufes)
FIG_DIR    = os.path.join("figures", "rq3"); os.makedirs(FIG_DIR, exist_ok=True)

LON, LAT = (-135.0, -115.0), (25.0, 48.0)
T0, T1 = pd.Timestamp("2012-01-01"), pd.Timestamp("2021-12-31")
EVENTS = {"Blob 2013-16": (pd.Timestamp("2013-01-01"), pd.Timestamp("2016-12-31")),
          "MHW 2019-20":  (pd.Timestamp("2019-01-01"), pd.Timestamp("2020-12-31"))}
EVENT_COLORS = {"Blob 2013-16": "#d55e00", "MHW 2019-20": "#0072b2"}

OISST_BASE = tuple(int(x) for x in os.environ.get("RQ3_BASE", "1982,2011").split(","))   # 30-year baseline, both events outside it
QUALITY_FLAG = 5            # flg_sst value treated as high quality
MIN_DAYS = 5                # Hobday: 5+ consecutive days
MIN_HOURS_PER_DAY = 12      # hourly SST values needed to make one drifter-day
DROGUED_ONLY = False        # SST does not depend on the drogue, so all drifters are used (more coverage). Set True to match RQ2's velocity work.

CLUSTER_GAP_D, CLUSTER_RADIUS, MIN_CELLS = 4, 1, 5   # event grouping rule
W_DAYS, BUF_CELLS = 30, 2                            # 'observed/detected' window (days) and buffer (cells, ~25 km each)
CAP_DAYS = 120                                       # max run length tracked per cell when measuring event duration
BOOT, SEED = 1000, 42
TEST_START, TEST_END = EVENTS["MHW 2019-20"][0], EVENTS["MHW 2019-20"][1]
RESULTS = {}

mpl.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "grid.linewidth": 0.6, "axes.titlesize": 11, "axes.labelsize": 9.5,
                     "legend.frameon": False, "font.size": 9.5})
def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=200, bbox_inches="tight"); plt.show()
def shade_events(ax, alpha=0.13):
    for n, (s, e) in EVENTS.items(): ax.axvspan(s, e, color=EVENT_COLORS[n], alpha=alpha, lw=0)
def hav_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.deg2rad(np.asarray(lon2) - np.asarray(lon1)) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))
# rough US West Coast (lat, lon), south to north, for map outlines only
COAST = np.array([[32.53,-117.12],[33.00,-117.28],[33.45,-117.70],[33.75,-118.40],[34.03,-118.55],[34.40,-119.50],[34.46,-120.47],
    [34.90,-120.65],[35.40,-120.90],[35.67,-121.30],[36.30,-121.90],[36.60,-121.95],[36.95,-122.05],[37.50,-122.50],[37.80,-122.50],
    [38.30,-123.05],[38.90,-123.70],[39.50,-123.80],[40.40,-124.40],[41.00,-124.10],[41.75,-124.20],[42.80,-124.55],[43.40,-124.30],
    [44.60,-124.07],[45.50,-123.97],[46.20,-124.05],[46.90,-124.10],[47.60,-124.40],[48.38,-124.70]])
print("OISST baseline:", OISST_BASE, "| figures ->", FIG_DIR)

OISST baseline: (1982, 2011) | figures -> figures/rq3


## Part 0 · Data: drifters, OISST and the 90th-percentile threshold

In [2]:
df = pd.read_pickle(GDP_FILE)
df["sst_ok"] = np.isfinite(df.sst) & (df.flg_sst == QUALITY_FLAG)
print(f"{len(df):,} hourly obs from {df.id.nunique()} drifters, {df.time.min():%Y-%m-%d} to {df.time.max():%Y-%m-%d}; "
      f"high-quality SST {df.sst_ok.mean():.1%}; drogued {df.drogued.mean():.1%}")

1,412,827 hourly obs from 341 drifters, 2012-01-01 to 2021-12-31; high-quality SST 96.1%; drogued 54.4%


In [3]:
def _normalise(da):
    da = da.rename({k: v for k, v in {"latitude": "lat", "longitude": "lon"}.items() if k in da.dims or k in da.coords})
    for z in ("zlev", "lev"):
        if z in da.dims: da = da.isel({z: 0}, drop=True)
    da = da.assign_coords(lon=((da["lon"] + 180) % 360) - 180).sortby("lon").sortby("lat")
    return da.sel(lat=slice(LAT[0] - 0.3, LAT[1] + 0.3), lon=slice(LON[0] - 0.3, LON[1] + 0.3))

import urllib.request, urllib.parse
ERDDAP_NC = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/ncdcOisst21Agg_LonPM180.nc"
def oisst_url(y):
    q = (f"sst[({y}-01-01T12:00:00Z):1:({y}-12-31T12:00:00Z)][(0.0):1:(0.0)]"
         f"[({LAT[0] - 0.375}):1:({LAT[1] + 0.375})][({LON[0] - 0.375}):1:({LON[1] + 0.375})]")
    return ERDDAP_NC + "?" + urllib.parse.quote(q, safe=":(),.-")
def oisst_ok(f):
    # a usable yearly file has (almost) every day; the old loader saved EMPTY (1982-90) and GAPPY (1992-98) files, which this catches
    try:
        with xr.open_dataarray(f) as da: return da.sizes.get("time", 0) >= 360
    except Exception: return False
def download_year(y, f):
    tmp = f + ".part"
    urllib.request.urlretrieve(oisst_url(y), tmp)
    with xr.open_dataset(tmp) as ds: da = _normalise(ds["sst"]).load()
    if da.sizes.get("time", 0) < 360: raise ValueError(f"NOAA returned only {da.sizes.get('time', 0)} days")
    da.to_netcdf(f); os.remove(tmp)

def load_oisst(years):
    os.makedirs(OISST_DIR, exist_ok=True); pieces, missing = [], []
    for y in years:
        f = os.path.join(OISST_DIR, f"oisst_{y}.nc")
        if not oisst_ok(f):
            print(f"OISST {y}: {'cached file is EMPTY/broken' if os.path.exists(f) else 'not cached'}, downloading ...", end=" ")
            try: download_year(y, f); print("ok")
            except Exception as e:
                missing.append(y); print(f"FAILED ({repr(e)[:90]})"); continue
        pieces.append(xr.open_dataarray(f).load().astype("float32"))
    if missing:
        print(f"\n!!! Could not get OISST for {missing}. Download each in a browser and save as {OISST_DIR}/oisst_YEAR.nc, e.g.:\n  {oisst_url(missing[0])}")
    da = xr.concat(pieces, dim="time").sortby("time")
    return da.assign_coords(time=pd.DatetimeIndex(da.time.values).floor("D")), missing

oisst, missing = load_oisst(range(OISST_BASE[0], T1.year + 1))
have = sorted(set(pd.DatetimeIndex(oisst.time.values).year))
base_have = [y for y in range(OISST_BASE[0], OISST_BASE[1] + 1) if y in have]
BASE_USED = (base_have[0], OISST_BASE[1])
if BASE_USED != OISST_BASE:
    print(f"!!! WARNING: baseline years missing; using {BASE_USED} instead of {OISST_BASE}. Numbers are PROVISIONAL until 1982-1990 are downloaded.")
RESULTS["OISST baseline used"] = f"{BASE_USED[0]}-{BASE_USED[1]}"
print(oisst.dims, oisst.shape, "| years", have[0], "-", have[-1])

OISST 1982: cached file is EMPTY/broken, downloading ... ok
OISST 1983: cached file is EMPTY/broken, downloading ... ok
OISST 1984: cached file is EMPTY/broken, downloading ... 

KeyboardInterrupt: 

In [ ]:
def hobday_threshold(da, base=BASE_USED, pct=90, half_win=5, smooth=31):
    # day-of-year percentile per grid cell (+-5 day window), then a 31-day circular moving average (Hobday et al. 2016)
    b = da.sel(time=slice(f"{base[0]}-01-01", f"{base[1]}-12-31")); arr = b.values
    doy = pd.DatetimeIndex(b.time.values).dayofyear.values
    thr = np.full((366,) + arr.shape[1:], np.nan, dtype=np.float32)
    for d in range(1, 367):
        diff = (doy - d + 183) % 366 - 183
        thr[d - 1] = np.nanpercentile(arr[np.abs(diff) <= half_win], pct, axis=0)
    k = smooth // 2; pad = np.concatenate([thr[-k:], thr, thr[:k]], axis=0)
    cs = np.cumsum(np.where(np.isnan(pad), 0, pad), axis=0, dtype=np.float64); cn = np.cumsum(~np.isnan(pad), axis=0)
    cs = np.concatenate([np.zeros((1,) + cs.shape[1:]), cs]); cn = np.concatenate([np.zeros((1,) + cn.shape[1:], int), cn])
    sm = (cs[smooth:] - cs[:-smooth]) / np.maximum(cn[smooth:] - cn[:-smooth], 1)
    sm[cn[smooth:] - cn[:-smooth] == 0] = np.nan
    return sm.astype(np.float32)

def persist_flag(exceed, n=MIN_DAYS):
    # True on every day that belongs to a run of >= n consecutive exceedance days, per grid cell
    fwd = np.zeros(exceed.shape, np.int16); bwd = np.zeros(exceed.shape, np.int16); run = np.zeros(exceed.shape[1:], np.int16)
    for t in range(exceed.shape[0]): run = np.where(exceed[t], run + 1, 0).astype(np.int16); fwd[t] = run
    run = np.zeros(exceed.shape[1:], np.int16)
    for t in range(exceed.shape[0] - 1, -1, -1): run = np.where(exceed[t], run + 1, 0).astype(np.int16); bwd[t] = run
    return exceed & ((fwd + bwd - 1) >= n)

thr = hobday_threshold(oisst)
study = oisst.sel(time=slice(T0, T1)); TAX = pd.DatetimeIndex(study.time.values)
sdoy = TAX.dayofyear.values; SST_S = study.values
exceed_grid = SST_S > thr[sdoy - 1]
mhw_grid = persist_flag(exceed_grid)
lat_g, lon_g = study.lat.values, study.lon.values
ocean = np.isfinite(thr[0])
print(f"2012-2021: {exceed_grid[:, ocean].mean():.1%} of ocean cell-days above P90 (10% expected in the baseline); "
      f"{mhw_grid[:, ocean].mean():.1%} in a 5+ day heatwave")

In [ ]:
# ---- drifter-days: daily mean SST per drifter, matched to the nearest OISST cell and day ----
m = df.sst_ok.copy()
if DROGUED_ONLY: m &= df.drogued
use = df[m].assign(date=lambda x: x.time.dt.floor("D"))
daily = (use.groupby(["id", "date"]).agg(sst=("sst", "mean"), lat=("lat", "mean"), lon=("lon", "mean"), n=("sst", "size"))
         .reset_index().query("n >= @MIN_HOURS_PER_DAY"))
daily["iy"] = np.abs(daily.lat.values[:, None] - lat_g[None, :]).argmin(1)
daily["ix"] = np.abs(daily.lon.values[:, None] - lon_g[None, :]).argmin(1)
daily["ti"] = TAX.get_indexer(daily.date)
daily = daily[daily.ti >= 0].copy(); daily["doy"] = daily.date.dt.dayofyear
daily["thr"] = thr[daily.doy.values - 1, daily.iy.values, daily.ix.values]
daily["oisst_sst"] = SST_S[daily.ti.values, daily.iy.values, daily.ix.values]
daily = daily[np.isfinite(daily.thr) & np.isfinite(daily.oisst_sst)].sort_values(["id", "date"]).reset_index(drop=True)
daily["oisst_mhw"] = mhw_grid[daily.ti.values, daily.iy.values, daily.ix.values]
bias = daily.sst - daily.oisst_sst
print(f"{len(daily):,} drifter-days from {daily.id.nunique()} drifters | drifter minus OISST SST: mean {bias.mean():+.2f} C, SD {bias.std():.2f} C")
RESULTS["Drifter-days"] = len(daily); RESULTS["Drifters"] = daily.id.nunique()

## Part A1 · Regional heatwave events (the unit of analysis)
Each OISST cell's heatwave run has a start day. Starts close in time (±4 days) and space (1 cell) are joined into one **regional event**. Each event gets a size (km² of cells that started), an intensity (mean °C above the threshold over its first 5 days), a duration (median run length of its cells) and an onset date (its first start).

In [ ]:
dlat = float(abs(np.median(np.diff(lat_g)))); dlon = float(abs(np.median(np.diff(lon_g))))
def build_events(mhw, sst, thr_arr, sdoy):
    T = mhw.shape[0]; prev = np.zeros_like(mhw); prev[1:] = mhw[:-1]
    onset = mhw & ~prev; onset[0] = False                       # runs already under way on day 1 are not onsets
    grown = ndimage.binary_dilation(onset, structure=np.ones((2 * CLUSTER_GAP_D + 1, 2 * CLUSTER_RADIUS + 1, 2 * CLUSTER_RADIUS + 1), bool))
    lab, _ = ndimage.label(grown, structure=np.ones((3, 3, 3), bool))
    tt, yy, xx = np.nonzero(onset); ev = lab[tt, yy, xx]
    alive = np.ones(len(tt), bool); dur = np.zeros(len(tt), int)
    for k in range(CAP_DAYS):
        alive = alive & (tt + k < T) & mhw[np.minimum(tt + k, T - 1), yy, xx]; dur += alive
    inten = np.zeros(len(tt))
    for k in range(MIN_DAYS):
        tk = np.minimum(tt + k, T - 1); inten += sst[tk, yy, xx] - thr_arr[sdoy[tk] - 1, yy, xx]
    inten /= MIN_DAYS
    P = pd.DataFrame({"ev": ev, "t": tt, "y": yy, "x": xx, "dur": dur, "inten": inten, "lat": lat_g[yy], "lon": lon_g[xx]})
    P["area"] = (111.32 * dlat) * (111.32 * dlon * np.cos(np.deg2rad(P.lat)))
    g = P.groupby("ev"); U = P.drop_duplicates(["ev", "y", "x"]).groupby("ev")
    E = pd.DataFrame({"n_cells": U.size(), "onset_t": g.t.min(), "area_km2": U.area.sum(), "intensity": g.inten.mean(),
                      "duration": g.dur.median(), "lat": g.lat.mean(), "lon": g.lon.mean()})
    E = E[E.n_cells >= MIN_CELLS].sort_values("onset_t").copy(); E["onset_date"] = TAX[E.onset_t.values]
    return E, P[P.ev.isin(E.index)]

def period_of(d):
    d = pd.Timestamp(d)
    for n, (s, e) in EVENTS.items():
        if s <= d <= e: return n
    return "other years"

EVT, PTS = build_events(mhw_grid, SST_S, thr, sdoy)
EVT["period"] = [period_of(d) for d in EVT.onset_date]
print(f"{len(EVT)} regional events (grouping: +-{CLUSTER_GAP_D} d, {CLUSTER_RADIUS} cell, >= {MIN_CELLS} cells)")
display(EVT.groupby("period").agg(events=("n_cells", "size"), median_area_km2=("area_km2", "median"), largest_area_km2=("area_km2", "max"),
                                  median_intensity_C=("intensity", "median"), median_duration_d=("duration", "median")).round(1))
RESULTS["Regional events"] = len(EVT)

## Part A2 · The real-time alert and matching alerts to events
The alert runs along each drifter as if data were arriving live: it fires on the **n-th consecutive day** above the percentile threshold, never earlier. Default = P90, 5 days (the same rule as RQ1 Model B plus persistence). Alternatives (P85/90/95 × 1/3/5/7 days) are computed for Part A5.

In [ ]:
def causal_alert(ids, dates, exc, n):
    # alert is on from the n-th consecutive day above the threshold (past data only); 'first' marks that n-th day
    exc = np.asarray(exc, bool); ids = pd.Series(ids); d = pd.Series(pd.DatetimeIndex(dates)); e = pd.Series(exc)
    brk = (ids != ids.shift()) | ((d - d.shift()).dt.days != 1) | (e != e.shift())
    pos = e.groupby(brk.cumsum()).cumcount().values + 1
    return exc & (pos >= n), exc & (pos == n)
_a, _f = causal_alert([1] * 9, pd.date_range("2020-01-01", periods=9), [0, 1, 1, 1, 1, 1, 1, 1, 0], 5)
assert _a.tolist() == [False] * 5 + [True, True, True, False] and _f.tolist() == [False] * 5 + [True] + [False] * 3
_a, _ = causal_alert([1] * 4 + [2] * 4, pd.date_range("2020-01-01", periods=8), [1] * 8, 5); assert not _a.any()
print("sanity checks passed: alert fires on day 5, never earlier; a new drifter restarts the count")

PCTS, NS = (85, 90, 95), (1, 3, 5, 7)
THR = {90: thr}
for p in PCTS:
    if p not in THR: THR[p] = hobday_threshold(oisst, pct=p)
D_ID, D_IY, D_IX, D_DATE, D_TI = daily.id.values, daily.iy.values, daily.ix.values, daily.date.values, daily.ti.values
ORDER = np.argsort(D_TI, kind="stable"); TI_SORTED = D_TI[ORDER]; NY, NX = mhw_grid.shape[1:]
ALERT, FIRST = {}, {}
for p in THR:
    exc_p = daily.sst.values > THR[p][daily.doy.values - 1, D_IY, D_IX]
    for n in NS: ALERT[(p, n)], FIRST[(p, n)] = causal_alert(D_ID, D_DATE, exc_p, n)

PTS_BY = {e: (g.y.values, g.x.values) for e, g in PTS.groupby("ev")}
def match_events(buf, W, E=None):
    E = EVT if E is None else E; out = {}; st = np.ones((2 * buf + 1, 2 * buf + 1), bool)
    for e, r in E.iterrows():
        mk = np.zeros((NY, NX), bool); y, x = PTS_BY[e]; mk[y, x] = True
        if buf > 0: mk = ndimage.binary_dilation(mk, structure=st)
        lo, hi = np.searchsorted(TI_SORTED, r.onset_t), np.searchsorted(TI_SORTED, r.onset_t + W, side="right")
        c = ORDER[lo:hi]; out[e] = c[mk[D_IY[c], D_IX[c]]]
    return out
def detect(A, matches, E=None):
    E = EVT if E is None else E; rows = []
    for e, r in E.iterrows():
        c = matches[e]; a = A[c] if len(c) else np.zeros(0, bool)
        lag = (D_DATE[c][a].min() - np.datetime64(r.onset_date)).astype("timedelta64[D]").astype(int) if a.any() else np.nan
        rows.append((e, len(c), len(np.unique(D_ID[c])), int(len(c) > 0), int(a.any()), lag))
    return pd.DataFrame(rows, columns=["event", "drifter_days", "drifters", "observed", "detected", "lag_d"]).set_index("event")

MATCH = match_events(BUF_CELLS, W_DAYS)
DEF = detect(ALERT[(90, MIN_DAYS)], MATCH).join(EVT[["onset_date", "period", "area_km2", "intensity", "duration", "lat", "lon", "n_cells"]])
assert (DEF.lag_d.dropna() >= 0).all()
print(f"{int(DEF.observed.sum())} of {len(DEF)} events had a drifter within ~{25 * BUF_CELLS} km in their first {W_DAYS} days; "
      f"{int(DEF[DEF.observed == 1].detected.sum())} of those were detected")
DEF.to_csv(os.path.join(FIG_DIR, "table_rq3_events.csv"))

## Part A3 · How often and how early? (default rule P90, 5 days)
Intervals are a bootstrap over **events** (events are the unit). "Other years" = events that started outside the two named windows (2012, 2017–18, 2021).

In [ ]:
rng = np.random.default_rng(SEED)
def boot_stat(v, fn, B=BOOT):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3: return (fn(v) if len(v) else np.nan), np.nan, np.nan
    b = np.array([fn(v[rng.integers(0, len(v), len(v))]) for _ in range(B)]); return fn(v), *np.percentile(b, [2.5, 97.5])
def fm(t, d=0, pct=False):
    p, lo, hi = t; s = (lambda z: f"{100 * z:.0f}%") if pct else (lambda z: f"{z:.{d}f}")
    return "n/a" if not np.isfinite(p) else (f"{s(p)} [{s(lo)}-{s(hi)}]" if np.isfinite(lo) else s(p))
def summarise(D):
    rows = {}
    for nm, g in [("All events", D)] + [(k, D[D.period == k]) for k in ("Blob 2013-16", "MHW 2019-20", "other years")]:
        o = g[g.observed == 1]
        rows[nm] = {"regional events": len(g),
                    "observed (drifter nearby)": fm(boot_stat(g.observed, np.mean), pct=True) if len(g) else "-",
                    "detected, of observed": fm(boot_stat(o.detected, np.mean), pct=True) if len(o) else "-",
                    "median lag, days": fm(boot_stat(o[o.detected == 1].lag_d, np.median), d=1) if (o.detected == 1).any() else "-",
                    "detected within 7 days, of observed": fm(boot_stat((o.lag_d <= 7).astype(float), np.mean), pct=True) if len(o) else "-"}
    return pd.DataFrame(rows).T
SUMM = summarise(DEF); display(SUMM); SUMM.to_csv(os.path.join(FIG_DIR, "table_rq3_detection_summary.csv"))
o = DEF[DEF.observed == 1]
print(f"Detector misses (drifter nearby, no alert): {int((o.detected == 0).sum())} | events no drifter observed: {int((DEF.observed == 0).sum())}")
RESULTS.update({"Events with a drifter nearby (%)": round(100 * DEF.observed.mean(), 1),
                "Detected, of observed (%)": round(100 * o.detected.mean(), 1) if len(o) else None,
                "Median lag, days": float(o[o.detected == 1].lag_d.median()) if (o.detected == 1).any() else None})

# are bigger events more likely to have a drifter nearby? (coverage vs size)
DEF["size_class"] = pd.qcut(DEF.area_km2, 4, labels=["smallest 25%", "2nd", "3rd", "largest 25%"], duplicates="drop")
SIZE = DEF.groupby("size_class", observed=True).agg(events=("observed", "size"), median_area_km2=("area_km2", "median"),
        observed_pct=("observed", lambda s: 100 * s.mean()),
        detected_pct_of_observed=("detected", lambda s: 100 * s[DEF.loc[s.index, "observed"] == 1].mean() if (DEF.loc[s.index, "observed"] == 1).any() else np.nan),
        median_lag_d=("lag_d", "median")).round(1)
display(SIZE); SIZE.to_csv(os.path.join(FIG_DIR, "table_rq3_by_size.csv"))

**How to read A3.** Three separate numbers, never merged: *coverage* (share of events with a drifter nearby), *reliability* (share of observed events that alerted) and *timeliness* (lag). A low coverage number is a property of the drifter array, not the detector. A median lag near 4–5 days means the alert is about as fast as a 5-day rule allows.
**Lag = 0** means a drifter in that spot was *already* alerting on the day OISST marked the onset (the water it was in had been warm for 5+ days). That is an early warning, not an error; say how many events had lag 0 when you quote the median.

## Part A4 · What is associated with detection and lag?
Spearman rank correlations (no model assumptions) with bootstrap intervals over events, then a leave-one-event-out logistic model: can event properties predict detection better than "always predict the usual detection rate"? If an interval includes 0, the data cannot show a relationship. Say so.

In [ ]:
FEATS = ["log_area", "intensity", "log_duration", "log_coverage"]
NAMES = {"log_area": "size (log km2)", "intensity": "intensity (C above P90)", "log_duration": "duration (log days)", "log_coverage": "coverage (log drifter-days)"}
DRV = DEF.copy(); DRV["log_area"] = np.log10(DRV.area_km2); DRV["log_duration"] = np.log10(DRV.duration.clip(lower=1))
DRV["log_coverage"] = np.log10(1 + DRV.drifter_days)
O = DRV[DRV.observed == 1].dropna(subset=FEATS).copy(); rows = []
for f in FEATS:
    for outcome, sub in (("lag (days, detected events)", O[O.detected == 1]), ("detected (0/1)", O)):
        y = sub.lag_d.values if outcome.startswith("lag") else sub.detected.values.astype(float); x = sub[f].values
        if len(sub) >= 8 and np.nanstd(y) > 0:
            rho = stats.spearmanr(x, y)[0]; b = []
            for _ in range(BOOT):
                i = rng.integers(0, len(sub), len(sub))
                if np.std(y[i]) > 0 and np.std(x[i]) > 0: b.append(stats.spearmanr(x[i], y[i])[0])
            rows.append({"property": NAMES[f], "outcome": outcome, "events": len(sub), "rho": rho,
                         "lo": np.percentile(b, 2.5) if len(b) > 20 else np.nan, "hi": np.percentile(b, 97.5) if len(b) > 20 else np.nan})
RHO = pd.DataFrame(rows)
if len(RHO):
    RHO.round(3).to_csv(os.path.join(FIG_DIR, "table_rq3_drivers.csv"), index=False)
    RHO["rho [95% CI]"] = RHO.apply(lambda r: fm((r.rho, r.lo, r.hi), d=2) + (" *" if (r.lo > 0 or r.hi < 0) else ""), axis=1)
    display(RHO.pivot_table(index="property", columns="outcome", values="rho [95% CI]", aggfunc="first"))
    print("* = interval excludes 0. Negative rho with lag = bigger/stronger/better-covered events are caught sooner.")
LOEO = None
if len(O) >= 20 and 4 <= O.detected.sum() <= len(O) - 4:
    Xo, yo = O[FEATS].values, O.detected.values; pr = np.zeros(len(O)); base = np.zeros(len(O))
    for i in range(len(O)):
        tr = np.arange(len(O)) != i
        mdl = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500)).fit(Xo[tr], yo[tr])
        pr[i] = mdl.predict_proba(Xo[i:i + 1])[0, 1]; base[i] = yo[tr].mean()
    bs = [roc_auc_score(yo[i], pr[i]) for i in (rng.integers(0, len(O), len(O)) for _ in range(BOOT)) if yo[i].min() != yo[i].max()]
    LOEO = dict(n=len(O), auc=roc_auc_score(yo, pr), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                brier_model=np.mean((pr - yo) ** 2), brier_base=np.mean((base - yo) ** 2))
    coef = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=500)).fit(Xo, yo)[-1].coef_[0]
    print(f"Leave-one-event-out ({LOEO['n']} observed events): AUC {LOEO['auc']:.2f} [{LOEO['lo']:.2f}-{LOEO['hi']:.2f}]; "
          f"Brier {LOEO['brier_model']:.3f} vs {LOEO['brier_base']:.3f} for the base rate (lower is better)")
    print("Standardised coefficients (positive = more likely detected):", {NAMES[k]: round(v, 2) for k, v in zip(FEATS, coef)})
    RESULTS["LOEO AUC"] = round(LOEO["auc"], 2)
else:
    print(f"Leave-one-event-out skipped: needs >= 20 observed events with >= 4 detected and >= 4 missed "
          f"(have {len(O)} observed, {int(O.detected.sum())} detected). Report this: too few misses to model what causes them.")

## Part A5 · Forward-in-time hold-out (tutor: "tune on earlier years, hold out 2019–20")
Every rule (P85/90/95 × 1/3/5/7 days) is scored on events that started **before Dec 2018** only; the best-F1 rule is chosen there, then tested **once** on the 2019–20 events, alongside the RQ1 default (P90, 5 d) and the no-persistence rule (P90, 1 d).
- **Recall** = share of observed events detected.
- **Precision** = share of first alerts that land on an OISST heatwave cell-day (false-alarm check).

In [ ]:
cut = TEST_START - pd.Timedelta(days=W_DAYS)
def rule_scores(p, n, ev_mask, ep_mask):
    D = detect(ALERT[(p, n)], MATCH); g = D[ev_mask.reindex(D.index).fillna(False).values]; o = g[g.observed == 1]
    ep = FIRST[(p, n)] & ep_mask; prec = float(daily.oisst_mhw.values[ep].mean()) if ep.sum() else np.nan
    rec = float(o.detected.mean()) if len(o) else np.nan
    f1 = 2 * prec * rec / (prec + rec) if np.isfinite(prec) and np.isfinite(rec) and prec + rec > 0 else np.nan
    return dict(events=len(g), observed=len(o), recall=rec, precision=prec, F1=f1, alerts=int(ep.sum()),
                median_lag=float(o[o.detected == 1].lag_d.median()) if (o.detected == 1).any() else np.nan)
tr_ev = pd.Series(EVT.onset_date.values < cut.to_datetime64(), index=EVT.index)
te_ev = pd.Series((EVT.onset_date.values >= TEST_START.to_datetime64()) & (EVT.onset_date.values <= TEST_END.to_datetime64()), index=EVT.index)
tr_ep = D_DATE < cut.to_datetime64(); te_ep = (D_DATE >= TEST_START.to_datetime64()) & (D_DATE <= TEST_END.to_datetime64())
rows = []
for (p, n) in ALERT:
    a, b = rule_scores(p, n, tr_ev, tr_ep), rule_scores(p, n, te_ev, te_ep)
    rows.append({"percentile": p, "days": n, **{f"train {k}": v for k, v in a.items()}, **{f"test {k}": v for k, v in b.items()}})
TUNE = pd.DataFrame(rows).set_index(["percentile", "days"]); TUNE.round(3).to_csv(os.path.join(FIG_DIR, "table_rq3_rule_holdout.csv"))
best = tuple(int(v) for v in TUNE["train F1"].idxmax())
SEL = pd.DataFrame({"rule": ["chosen on pre-2019 events", "RQ1 default (P90, 5 d)", "no persistence (P90, 1 d)"],
                    "percentile, days": [best, (90, MIN_DAYS), (90, 1)]})
for k in ("train F1", "test events", "test observed", "test recall", "test precision", "test F1", "test median_lag"):
    SEL[k] = [TUNE.loc[r, k] for r in SEL["percentile, days"]]
display(SEL.set_index("rule").round(2))
print(f"Training: {int(tr_ev.sum())} events before {cut.date()} | test: {int(te_ev.sum())} events in 2019-20. The test set was not used to choose the rule.")
RESULTS["Rule chosen on pre-2019 events"] = str(best)
RESULTS["Test F1 (chosen rule)"] = round(float(TUNE.loc[best, "test F1"]), 2)
RESULTS["Test F1 (P90, 5 d)"] = round(float(TUNE.loc[(90, MIN_DAYS), "test F1"]), 2)
with pd.option_context("display.max_columns", 30):
    display(TUNE[["train recall", "train precision", "train F1", "train median_lag", "test recall", "test precision", "test F1", "test median_lag"]].round(2))

**How to read A5.** Longer persistence usually trades lag for precision: 1-day rules alert sooner but many first alerts are not real heatwave days. If the pre-2019 choice is (or ties with) P90/5 d and it holds up on 2019–20, that justifies the RQ1 threshold *and* duration with data, which is exactly what the tutor asked.

## Part A6 · Sensitivity: does the picture change with how we define 'near', 'soon' and 'one event'?

In [ ]:
rows = []
for buf in (0, 1, 2, 3):
    for W in (15, 30, 60):
        D = detect(ALERT[(90, MIN_DAYS)], match_events(buf, W)); o_ = D[D.observed == 1]
        rows.append({"buffer (km)": 25 * buf, "window (d)": W, "observed %": 100 * D.observed.mean(),
                     "detected % of observed": 100 * o_.detected.mean() if len(o_) else np.nan,
                     "median lag (d)": o_[o_.detected == 1].lag_d.median() if (o_.detected == 1).any() else np.nan})
SENS = pd.DataFrame(rows).round(1); display(SENS.set_index(["buffer (km)", "window (d)"]))
SENS.to_csv(os.path.join(FIG_DIR, "table_rq3_sensitivity_buffer_window.csv"), index=False)

rows = []
_saved = (CLUSTER_GAP_D, CLUSTER_RADIUS, MIN_CELLS)
for cg, cr, mc in ((4, 1, 5), (2, 1, 5), (7, 1, 5), (4, 2, 5), (4, 1, 3), (4, 1, 10)):
    CLUSTER_GAP_D, CLUSTER_RADIUS, MIN_CELLS = cg, cr, mc
    E2, P2 = build_events(mhw_grid, SST_S, thr, sdoy)
    _pts_saved = PTS_BY; PTS_BY = {e: (g.y.values, g.x.values) for e, g in P2.groupby("ev")}
    D2 = detect(ALERT[(90, MIN_DAYS)], match_events(BUF_CELLS, W_DAYS, E2), E2); o2 = D2[D2.observed == 1]
    PTS_BY = _pts_saved
    rows.append({"time gap (d)": cg, "space gap (cells)": cr, "min cells": mc, "regional events": len(E2),
                 "median area km2": E2.area_km2.median(), "observed %": 100 * D2.observed.mean(),
                 "detected % of observed": 100 * o2.detected.mean() if len(o2) else np.nan,
                 "median lag (d)": o2[o2.detected == 1].lag_d.median() if (o2.detected == 1).any() else np.nan})
CLUSTER_GAP_D, CLUSTER_RADIUS, MIN_CELLS = _saved
GRP = pd.DataFrame(rows).round(1); display(GRP); GRP.to_csv(os.path.join(FIG_DIR, "table_rq3_sensitivity_grouping.csv"), index=False)
print("First row = default. Event COUNTS depend on the grouping rule, so always quote the rule next to a count; "
      "the detection % and lag should be what stays stable.")

## Part A7 · Figures for the slides

In [ ]:
# Fig A: every regional event through time: size, and whether it was detected / missed / not observed
st = np.where(DEF.observed == 0, "not observed", np.where(DEF.detected == 1, "detected", "missed"))
fig, ax = plt.subplots(figsize=(12.5, 4.4))
for k, c, mk in (("not observed", "#bdbdbd", "s"), ("detected", "#009e73", "o"), ("missed", "#d55e00", "X")):
    s = st == k
    ax.scatter(DEF.onset_date[s], DEF.area_km2[s], c=c, marker=mk, s=16 + 20 * np.clip(DEF.intensity[s].fillna(0), 0, 3),
               label=f"{k} (n={int(s.sum())})", alpha=0.85, edgecolor="k", lw=0.3)
ax.set_yscale("log"); ax.set_ylabel("event size (km², log scale)"); shade_events(ax)
ax.legend(fontsize=8.5, loc="upper left", ncol=3)
ax.set_title("Regional marine heatwave events (OISST, 2012–2021) and whether a drifter alert caught them  ·  marker size = intensity")
fig.tight_layout(); savefig(fig, "figA_events_timeline")

In [ ]:
# Fig B: coverage vs reliability by event size (the key 'size' result as a simple bar chart)
fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(len(SIZE)); w = 0.38
b1 = ax.bar(x - w / 2, SIZE.observed_pct, w, color="#9aa0a6", label="had a drifter nearby (coverage)")
b2 = ax.bar(x + w / 2, SIZE.detected_pct_of_observed, w, color="#009e73", label="detected, of those observed (reliability)")
for bars in (b1, b2):
    for r_ in bars:
        if np.isfinite(r_.get_height()): ax.text(r_.get_x() + r_.get_width() / 2, r_.get_height() + 1.5, f"{r_.get_height():.0f}%", ha="center", fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels([f"{i}\n(median {a:,.0f} km²)" for i, a in zip(SIZE.index, SIZE.median_area_km2)], fontsize=8.5)
ax.set_ylim(0, 112); ax.set_ylabel("% of events"); ax.legend(fontsize=8.5, loc="upper left")
ax.set_title("Bigger heatwaves are more likely to be sampled — is the detector also more reliable?")
fig.tight_layout(); savefig(fig, "figB_coverage_vs_size")

In [ ]:
# Fig C: lag against event properties (detected events)
Od = DEF[(DEF.observed == 1) & (DEF.detected == 1)]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, (f, lab, lg) in zip(axes, (("area_km2", "event size (km²)", True), ("intensity", "intensity (°C above P90)", False),
                                   ("duration", "duration (days)", True), ("drifter_days", "drifter-days nearby (coverage)", True))):
    ax.scatter(Od[f], Od.lag_d, s=22, color="#0072b2", alpha=0.7, edgecolor="k", lw=0.3)
    ax.axhline(MIN_DAYS - 1, color="#999", ls="--", lw=0.9)
    if lg: ax.set_xscale("log")
    rho = stats.spearmanr(Od[f], Od.lag_d)[0] if len(Od) >= 8 else np.nan
    ax.set_xlabel(lab); ax.set_title(f"Spearman ρ = {rho:.2f}" if np.isfinite(rho) else "too few events", fontsize=9.5)
axes[0].set_ylabel("days from onset to first alert")
fig.suptitle("Detection lag vs event properties (each dot = one detected regional event; dashed = fastest a 5-day rule can be)", y=1.03)
fig.tight_layout(); savefig(fig, "figC_lag_drivers")

In [ ]:
# Fig D: rule grid, F1 on pre-2019 events (left) and on held-out 2019-20 events (right)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
for ax, col, ttl in ((axes[0], "train F1", "Choose: pre-2019 events"), (axes[1], "test F1", "Test once: 2019–20 events")):
    piv = TUNE[col].unstack(); im = ax.imshow(piv.values, cmap="viridis", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels([f"P{p}" for p in piv.index]); ax.set_xlabel("days above threshold before alerting"); ax.grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if np.isfinite(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="w" if v < 0.6 else "k", fontsize=8.5)
    bi = list(piv.index).index(best[0]); bj = list(piv.columns).index(best[1])
    ax.add_patch(plt.Rectangle((bj - .5, bi - .5), 1, 1, fill=False, ec="#d55e00", lw=2.2)); ax.set_title(ttl)
fig.colorbar(im, ax=axes, shrink=0.85, label="F1 (recall of events × precision of alerts)")
fig.suptitle("Alert rule chosen on earlier years (orange box), then tested on the held-out 2019–20 heatwave", y=1.04)
savefig(fig, "figD_rule_holdout")

# Part B · Does exposure relate to where fish were? (answers the tutor's question)
**Data.** NOAA CalCOFI **CUFES** (Continuous Underway Fish Egg Sampler): ship surveys that pump surface water and count fish eggs, with ship SST at each sample (`data/cufes_cc.csv`, ERDDAP `erdCalCOFIcufes`). Eggs mark **where adults spawned**, so this is an observation of where fish were, at the time and place of the heatwaves.

**What it can and cannot show.** Association, not cause: the sardine stock was already declining (fishery closed 2015), and surveys sample different areas in different years. So every comparison is made **like for like**: spring cruises (Mar–May) and the same latitude bands. 2019–20 coverage is thin (Feb/Apr 2019 south of 35°N; Jan 2020), so that event cannot be tested properly. Say so on the slide.

In [ ]:
cu = pd.read_csv(CUFES_FILE)
cu = cu[pd.to_numeric(cu.latitude, errors="coerce").notna()].copy()
for c in ["latitude", "longitude", "start_temperature", "sardine_eggs", "anchovy_eggs"]: cu[c] = pd.to_numeric(cu[c], errors="coerce")
cu["time"] = pd.to_datetime(cu.time, utc=True).dt.tz_localize(None); cu["date"] = cu.time.dt.floor("D")
cu["y"], cu["m"] = cu.time.dt.year, cu.time.dt.month
cu["sardine"], cu["anchovy"] = cu.sardine_eggs > 0, cu.anchovy_eggs > 0
def per(t):
    if t < pd.Timestamp("2014-01-01"): return "1 Before (2012-13)"
    if t < pd.Timestamp("2017-01-01"): return "2 Blob peak (2014-16)"
    if t < pd.Timestamp("2019-01-01"): return "3 Between (2017-18)"
    if t < pd.Timestamp("2021-01-01"): return "4 MHW (2019-20)"
    return "5 After (2021)"
cu["period"] = cu.time.map(per)
spring = cu[cu.m.isin([3, 4, 5])].copy()
print(f"{len(cu):,} CUFES samples {cu.time.min():%Y-%m} to {cu.time.max():%Y-%m}; spring (Mar-May): {len(spring):,}")

# heatwave exposure of every sample: is its OISST cell in a 5+ day heatwave that day?
iy = np.abs(cu.latitude.values[:, None] - lat_g[None, :]).argmin(1); ix = np.abs(cu.longitude.values[:, None] - lon_g[None, :]).argmin(1)
ti = TAX.get_indexer(cu.date); okk = ti >= 0
cu["in_mhw"] = False; cu.loc[okk, "in_mhw"] = mhw_grid[ti[okk], iy[okk], ix[okk]]
spring = cu[cu.m.isin([3, 4, 5])].copy()

def boot_share(g, col, B=BOOT):
    # share of samples with eggs; 95% CI from resampling survey DAYS (samples on one day are not independent)
    days = g.date.values; u = np.unique(days); k = {d: np.flatnonzero(days == d) for d in u}; v = g[col].values.astype(float)
    if len(u) < 3: return v.mean(), np.nan, np.nan
    b = [v[np.concatenate([k[d] for d in rng.choice(u, len(u))])].mean() for _ in range(B)]
    return v.mean(), *np.percentile(b, [2.5, 97.5])

### B1 · Same season, same place: southern/central California core (31.5–35.5°N), spring

In [ ]:
core = spring[spring.latitude.between(31.5, 35.5)]
rows = []
for p_, g in core.groupby("period"):
    s_ = boot_share(g, "sardine"); a_ = boot_share(g, "anchovy")
    rows.append({"period": p_[2:], "samples": len(g), "survey days": g.date.nunique(), "ship SST (median °C)": g.start_temperature.median(),
                 "in OISST heatwave (%)": 100 * g.in_mhw.mean(),
                 "sardine eggs present": fm(s_, pct=True), "anchovy eggs present": fm(a_, pct=True)})
B1 = pd.DataFrame(rows).set_index("period"); display(B1.round(1)); B1.to_csv(os.path.join(FIG_DIR, "table_fish_B1_core_spring.csv"))
print("Intervals: bootstrap over survey days.")

### B2 · Did sardine spawning move north? (only bands that were actually surveyed count)

In [ ]:
spring["band"] = pd.cut(spring.latitude, [29, 32, 35, 38, 41, 44, 48], labels=["29-32°N", "32-35°N", "35-38°N", "38-41°N", "41-44°N", "44-48°N"])
n_tab = spring.pivot_table(index="y", columns="band", values="sardine", aggfunc="size", observed=False).fillna(0).astype(int)
s_tab = spring.pivot_table(index="y", columns="band", values="sardine", aggfunc="mean", observed=False)
print("Spring samples per latitude band (0 = not surveyed that year):"); display(n_tab)
print("Share of spring samples with sardine eggs (blank = not surveyed):"); display((100 * s_tab).round(0))
pos = spring[spring.sardine]
B2 = pos.groupby("period").agg(positive_samples=("sardine", "size"), median_lat=("latitude", "median"),
                                egg_weighted_lat=("latitude", lambda s: np.average(s, weights=pos.loc[s.index, "sardine_eggs"])),
                                ship_SST_at_eggs=("start_temperature", "median"))
B2.index = [i[2:] for i in B2.index]; display(B2.round(2)); B2.to_csv(os.path.join(FIG_DIR, "table_fish_B2_sardine_latitude.csv"))
print("Fair comparison: northern bands (>=38N) were surveyed in spring 2012, 2015 and 2016 only; compare those years.")

### B3 · Direct link: were eggs less likely in samples taken inside an OISST heatwave? (same band, same season)

In [ ]:
rows = []
for sp_ in ("sardine", "anchovy"):
    for band, g in spring.groupby("band", observed=True):
        if g.in_mhw.sum() >= 20 and (~g.in_mhw).sum() >= 20:
            a, b = g[g.in_mhw], g[~g.in_mhw]
            rows.append({"species": sp_, "band": band, "samples in heatwave": len(a), "samples not in heatwave": len(b),
                         "eggs present, in heatwave %": 100 * a[sp_].mean(), "eggs present, not in heatwave %": 100 * b[sp_].mean(),
                         "difference (pp)": 100 * (a[sp_].mean() - b[sp_].mean()),
                         "Fisher p": stats.fisher_exact([[a[sp_].sum(), len(a) - a[sp_].sum()], [b[sp_].sum(), len(b) - b[sp_].sum()]])[1]})
B3 = pd.DataFrame(rows).round(3); display(B3); B3.to_csv(os.path.join(FIG_DIR, "table_fish_B3_eggs_in_vs_out_heatwave.csv"), index=False)
print("Within a band and season, this compares samples taken inside vs outside an OISST heatwave. p-values treat samples as independent, "
      "which they are not (same cruise), so read them as indicative.")

### B4 · Do drifter temperatures match what the ship measured? (validates using drifters for fish exposure)

In [ ]:
pairs = []
dd = daily[["id", "date", "sst", "lat", "lon"]]
for shift in (-1, 0, 1):
    q = cu[["date", "latitude", "longitude", "start_temperature"]].assign(date=lambda x: x.date + pd.Timedelta(days=shift)).reset_index()
    mm = q.merge(dd, on="date")
    mm["km"] = hav_km(mm.latitude.values, mm.longitude.values, mm.lat.values, mm.lon.values)
    pairs.append(mm[mm.km <= 50])
PR = pd.concat(pairs).sort_values("km").drop_duplicates("index")
if len(PR) >= 10:
    diff = PR.sst - PR.start_temperature
    print(f"{len(PR)} CUFES samples with a drifter within 50 km and 1 day: drifter minus ship SST mean {diff.mean():+.2f} °C, "
          f"SD {diff.std():.2f} °C, r = {np.corrcoef(PR.sst, PR.start_temperature)[0, 1]:.2f}")
    RESULTS["Drifter vs ship SST pairs"] = len(PR); RESULTS["Drifter minus ship SST (C)"] = round(float(diff.mean()), 2)
else: print("Too few drifter-ship matches within 50 km and 1 day.")

In [ ]:
# Fig E: the one fish slide: (a,b) where sardine spawned before vs during the Blob (only surveyed areas shown), (c) southern core by year
fig = plt.figure(figsize=(14, 5.2)); gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1.5])
maps = [("Spring 2012 (before the Blob)", spring[spring.y == 2012]), ("Spring 2015–16 (Blob peak)", spring[spring.y.isin([2015, 2016])])]
for k, (ttl, g) in enumerate(maps):
    ax = fig.add_subplot(gs[k]); ax.plot(COAST[:, 1], COAST[:, 0], color="#777", lw=0.9)
    ax.scatter(g.longitude, g.latitude, s=3, color="#d0d0d0", label="surveyed, no sardine eggs")
    p = g[g.sardine]; ax.scatter(p.longitude, p.latitude, s=8 + 3 * np.sqrt(p.sardine_eggs), color="#0072b2", alpha=0.75, edgecolor="k", lw=0.2,
                                  label="sardine eggs (size = count)")
    ax.set_xlim(-129.5, -116.5); ax.set_ylim(29.5, 48.2); ax.set_aspect(1 / np.cos(np.deg2rad(38)))
    ax.set_title(f"{ttl}\nmedian latitude of eggs: {p.latitude.median():.1f}°N", fontsize=10); ax.set_xlabel("longitude")
    if k == 0: ax.set_ylabel("latitude"); ax.legend(fontsize=7.5, loc="lower left")
ax = fig.add_subplot(gs[2]); yr = core.groupby("y").agg(sard=("sardine", "mean"), anch=("anchovy", "mean"), sst=("start_temperature", "median"))
ax.axvspan(2013.5, 2016.5, color=EVENT_COLORS["Blob 2013-16"], alpha=0.13, lw=0); ax.axvspan(2018.5, 2020.5, color=EVENT_COLORS["MHW 2019-20"], alpha=0.13, lw=0)
ax.plot(yr.index, 100 * yr.sard, "o-", color="#0072b2", lw=2, label="sardine eggs present (%)")
ax.plot(yr.index, 100 * yr.anch, "s-", color="#e69f00", lw=2, label="anchovy eggs present (%)")
ax.set_ylabel("% of spring samples, 31.5–35.5°N"); ax.set_xticks(yr.index); ax.legend(fontsize=8, loc="upper left")
ax2 = ax.twinx(); ax2.plot(yr.index, yr.sst, "--", color="#d55e00", lw=1.4, label="ship SST (median)"); ax2.set_ylabel("ship SST (°C)", color="#d55e00")
ax2.grid(False); ax2.spines["right"].set_visible(True)
ax.set_title("Southern/central California, spring only (same place, same season)", fontsize=10)
fig.suptitle("Does heatwave exposure relate to where fish were?  CalCOFI egg surveys (spawning locations), 2012–2021", y=1.02, fontsize=12)
fig.tight_layout(); savefig(fig, "figE_fish_eggs")

## Part C · Headline numbers (copy into the report)

In [ ]:
display(pd.Series(RESULTS, name="value").to_frame())
pd.Series(RESULTS, name="value").to_csv(os.path.join(FIG_DIR, "rq3_headline_numbers.csv"))
print("Saved to", FIG_DIR, ":", sorted(os.listdir(FIG_DIR)))

### Limitations to state for RQ3
- **Truth is OISST, which already assimilates drifter SST.** Detection and lag measure agreement with OISST onset, not independent truth.
- **Coverage is opportunistic.** Drifters were not deployed to watch heatwaves, so many events have no drifter nearby; that is reported separately from detector misses.
- **Event counts depend on the grouping rule** (Part A6); detection % and lag are the quantities to quote.
- **Only two named events, one held-out period**, so intervals are wide.
- **Fish extension:** eggs show spawning location only; spring surveys only; sardine was already declining; 2019–20 egg coverage is too thin to test. Association, not cause.